## 1. Setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import joblib
import json
from pathlib import Path

from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import (
    classification_report, confusion_matrix, f1_score,
    average_precision_score, roc_auc_score,
    ConfusionMatrixDisplay, PrecisionRecallDisplay
)

import lightgbm as lgb
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    AutoModel, TrainingArguments, Trainer
)
import shap

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline

RANDOM_STATE = 42
N_FOLDS = 5
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

Device: cpu


---
## 2. Load Data

In [2]:
df = pd.read_csv(Path('../../data/claims_cleaned.csv'))

# Recover raw "other" text column
df_raw = pd.read_excel(Path('../../claim_use_case_dataset.xlsx'), engine='openpyxl')
other_text = df_raw['other'].fillna('').astype(str)
texts = (df['issueDesc'].fillna('') + ' ' + other_text).str.strip().tolist()
y = df['target'].values

# Structural features (same as v2)
DROP_FEATURES = [
    'deviceCost', 'balance_change', 'has_balance_change', 'oldBalanceRRP',
    'balanceRRP', 'smashed', 'frontOrBackCamera', 'other',
    'relationship_encoded', 'buttons', 'connection', 'charging',
]
with open(Path('../../data/feature_columns.txt'), 'r') as f:
    all_features = [line.strip() for line in f.readlines()]
structural_features = [f for f in all_features if f not in DROP_FEATURES]
X_struct = df[structural_features].fillna(0).values

print(f'Texts: {len(texts)}, Structural features: {X_struct.shape}')
print(f'Target: {np.bincount(y)} (0=Declined, 1=Completed)')

Texts: 2880, Structural features: (2880, 33)
Target: [ 453 2427] (0=Declined, 1=Completed)


In [3]:
# Shared evaluation function
def summarize_results(y_true, y_pred, y_proba, model_name):
    """Compute and print all metrics for a model."""
    f1_m = f1_score(y_true, y_pred, average='macro')
    f1_d = f1_score(y_true, y_pred, pos_label=0)
    pr_auc = average_precision_score(y_true, y_proba)
    roc = roc_auc_score(y_true, y_proba)
    
    print(f'\n{"="*60}')
    print(f'{model_name}')
    print(f'{"="*60}')
    print(f'  F1 Macro:    {f1_m:.4f}')
    print(f'  F1 Declined: {f1_d:.4f}')
    print(f'  PR-AUC:      {pr_auc:.4f}')
    print(f'  ROC-AUC:     {roc:.4f}')
    print(classification_report(y_true, y_pred, target_names=['Declined', 'Completed']))
    
    return {'f1_macro': f1_m, 'f1_declined': f1_d, 'pr_auc': pr_auc, 'roc_auc': roc}


def cv_collect(model_fn, X_data, y_data, model_name, n_folds=N_FOLDS):
    """Generic CV loop. model_fn(X_train, y_train, X_val) -> (y_pred, y_proba)."""
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=RANDOM_STATE)
    all_true, all_pred, all_proba = [], [], []
    fold_metrics = []
    
    for fold, (tr_idx, va_idx) in enumerate(skf.split(np.zeros(len(y_data)), y_data)):
        y_pred, y_proba = model_fn(X_data, y_data, tr_idx, va_idx)
        y_val = y_data[va_idx]
        
        fold_metrics.append({
            'fold': fold+1,
            'f1_macro': f1_score(y_val, y_pred, average='macro'),
            'f1_declined': f1_score(y_val, y_pred, pos_label=0),
            'pr_auc': average_precision_score(y_val, y_proba),
            'roc_auc': roc_auc_score(y_val, y_proba),
        })
        all_true.extend(y_val); all_pred.extend(y_pred); all_proba.extend(y_proba)
        print(f'  Fold {fold+1}: F1m={fold_metrics[-1]["f1_macro"]:.3f}, F1d={fold_metrics[-1]["f1_declined"]:.3f}')
    
    mdf = pd.DataFrame(fold_metrics)
    all_true, all_pred, all_proba = np.array(all_true), np.array(all_pred), np.array(all_proba)
    metrics = summarize_results(all_true, all_pred, all_proba, model_name)
    
    print(f'Mean +/- Std:')
    for col in ['f1_macro', 'f1_declined', 'pr_auc', 'roc_auc']:
        print(f'  {col:15s}: {mdf[col].mean():.4f} +/- {mdf[col].std():.4f}')
    
    return {'metrics': metrics, 'metrics_df': mdf, 'y_true': all_true, 'y_pred': all_pred, 'y_proba': all_proba}

---
## 3. Approach A: TF-IDF + SVD + LightGBM

The simplest NLP approach. TF-IDF captures **which words appear** and how distinctive they are. Unlike sentence embeddings, it preserves individual vocabulary signals (e.g., "stolen", "crack", "warranty").

In [4]:
def tfidf_lgb_fn(X_data, y_data, tr_idx, va_idx):
    texts_arr = np.array(texts)
    texts_tr, texts_va = texts_arr[tr_idx], texts_arr[va_idx]
    y_tr = y_data[tr_idx]
    
    # TF-IDF: word (1-2 gram) + char (2-4 gram)
    tfidf_word = TfidfVectorizer(max_features=3000, ngram_range=(1, 2), sublinear_tf=True)
    tfidf_char = TfidfVectorizer(max_features=3000, analyzer='char_wb', ngram_range=(2, 4), sublinear_tf=True)
    
    X_word_tr = tfidf_word.fit_transform(texts_tr)
    X_word_va = tfidf_word.transform(texts_va)
    X_char_tr = tfidf_char.fit_transform(texts_tr)
    X_char_va = tfidf_char.transform(texts_va)
    
    from scipy.sparse import hstack
    X_tfidf_tr = hstack([X_word_tr, X_char_tr])
    X_tfidf_va = hstack([X_word_va, X_char_va])
    
    # SVD reduction: 6000 -> 80
    svd = TruncatedSVD(n_components=80, random_state=RANDOM_STATE)
    X_svd_tr = svd.fit_transform(X_tfidf_tr)
    X_svd_va = svd.transform(X_tfidf_va)
    
    # Combine with structural features
    X_combined_tr = np.hstack([X_struct[tr_idx], X_svd_tr])
    X_combined_va = np.hstack([X_struct[va_idx], X_svd_va])
    
    model = lgb.LGBMClassifier(
        n_estimators=500, max_depth=5, learning_rate=0.05,
        is_unbalance=True, random_state=RANDOM_STATE, n_jobs=-1, verbose=-1,
    )
    model.fit(X_combined_tr, y_tr)
    y_pred = model.predict(X_combined_va)
    y_proba = model.predict_proba(X_combined_va)[:, 1]
    return y_pred, y_proba

print('=== Approach A: TF-IDF + SVD + LightGBM ===')
tfidf_results = cv_collect(tfidf_lgb_fn, None, y, 'TF-IDF + SVD + LightGBM')

=== Approach A: TF-IDF + SVD + LightGBM ===
  Fold 1: F1m=0.585, F1d=0.267
  Fold 2: F1m=0.577, F1d=0.269
  Fold 3: F1m=0.632, F1d=0.360
  Fold 4: F1m=0.576, F1d=0.252
  Fold 5: F1m=0.586, F1d=0.276

TF-IDF + SVD + LightGBM
  F1 Macro:    0.5918
  F1 Declined: 0.2857
  PR-AUC:      0.8979
  ROC-AUC:     0.6717
              precision    recall  f1-score   support

    Declined       0.38      0.23      0.29       453
   Completed       0.87      0.93      0.90      2427

    accuracy                           0.82      2880
   macro avg       0.63      0.58      0.59      2880
weighted avg       0.79      0.82      0.80      2880

Mean +/- Std:
  f1_macro       : 0.5912 +/- 0.0232
  f1_declined    : 0.2847 +/- 0.0430
  pr_auc         : 0.9002 +/- 0.0142
  roc_auc        : 0.6731 +/- 0.0319


---
## 4. Approach B: BiLSTM on Text

A sequence model that processes text word-by-word. Randomly initialized embeddings — no pretrained knowledge.
This tests whether **word order matters** for classification.

**Expected:** Weakest of the three due to small data (2,880) and multilingual vocabulary.

In [5]:
# Build a simple word-level vocabulary from all texts
from collections import Counter

MAX_VOCAB = 8000
MAX_LEN = 128

# Simple whitespace tokenizer
all_words = []
for t in texts:
    all_words.extend(t.lower().split())

word_counts = Counter(all_words)
vocab = {w: i+2 for i, (w, _) in enumerate(word_counts.most_common(MAX_VOCAB))}
vocab['<PAD>'] = 0
vocab['<UNK>'] = 1

def tokenize_text(text):
    tokens = text.lower().split()[:MAX_LEN]
    ids = [vocab.get(w, 1) for w in tokens]
    # Pad
    ids = ids + [0] * (MAX_LEN - len(ids))
    return ids

all_token_ids = np.array([tokenize_text(t) for t in texts])
print(f'Vocabulary size: {len(vocab)}')
print(f'Token matrix: {all_token_ids.shape}')

Vocabulary size: 8002
Token matrix: (2880, 128)


In [6]:
class ClaimDataset(Dataset):
    def __init__(self, token_ids, labels):
        self.token_ids = torch.LongTensor(token_ids)
        self.labels = torch.FloatTensor(labels)
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        return self.token_ids[idx], self.labels[idx]


class BiLSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, hidden_dim=64, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True, num_layers=1)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, 1)
    
    def forward(self, x):
        emb = self.embedding(x)
        lstm_out, (h_n, _) = self.lstm(emb)
        # Concatenate forward and backward final hidden states
        hidden = torch.cat((h_n[0], h_n[1]), dim=1)
        hidden = self.dropout(hidden)
        return self.fc(hidden).squeeze(-1)

In [7]:
def train_lstm(token_ids_tr, y_tr, token_ids_va, y_va, epochs=15, lr=1e-3, batch_size=64):
    """Train BiLSTM and return predictions."""
    # Class weights for imbalance
    pos_weight = torch.FloatTensor([len(y_tr[y_tr==0]) / max(len(y_tr[y_tr==1]), 1)]).to(device)
    
    model = BiLSTMClassifier(len(vocab)).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    
    train_ds = ClaimDataset(token_ids_tr, y_tr)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            optimizer.zero_grad()
            logits = model(batch_x)
            loss = criterion(logits, batch_y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
    
    # Predict
    model.eval()
    with torch.no_grad():
        val_x = torch.LongTensor(token_ids_va).to(device)
        logits = model(val_x)
        proba = torch.sigmoid(logits).cpu().numpy()
    
    y_pred = (proba >= 0.5).astype(int)
    return y_pred, proba


def lstm_fn(X_data, y_data, tr_idx, va_idx):
    return train_lstm(all_token_ids[tr_idx], y_data[tr_idx],
                      all_token_ids[va_idx], y_data[va_idx])

print('=== Approach B: BiLSTM ===')
lstm_results = cv_collect(lstm_fn, None, y, 'BiLSTM (text only)')

=== Approach B: BiLSTM ===
  Fold 1: F1m=0.530, F1d=0.218
  Fold 2: F1m=0.549, F1d=0.261
  Fold 3: F1m=0.543, F1d=0.276
  Fold 4: F1m=0.584, F1d=0.298
  Fold 5: F1m=0.549, F1d=0.285

BiLSTM (text only)
  F1 Macro:    0.5517
  F1 Declined: 0.2682
  PR-AUC:      0.8739
  ROC-AUC:     0.5992
              precision    recall  f1-score   support

    Declined       0.23      0.31      0.27       453
   Completed       0.86      0.81      0.84      2427

    accuracy                           0.73      2880
   macro avg       0.55      0.56      0.55      2880
weighted avg       0.76      0.73      0.75      2880

Mean +/- Std:
  f1_macro       : 0.5511 +/- 0.0199
  f1_declined    : 0.2675 +/- 0.0310
  pr_auc         : 0.8744 +/- 0.0115
  roc_auc        : 0.5982 +/- 0.0201


---
## 5. Approach C: Fine-tuned Multilingual BERT (Text Only)

Using `distilbert-base-multilingual-cased` — a smaller, faster multilingual BERT:
- 134M params, 6 transformer layers
- Pretrained on 104 languages including Swedish, Dutch, Finnish
- Fine-tune classification head + last layers on our data

In [8]:
BERT_MODEL = 'distilbert-base-multilingual-cased'
BERT_MAX_LEN = 128
BERT_EPOCHS = 3
BERT_BATCH = 16
BERT_LR = 2e-5

tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL)
print(f'Tokenizer loaded: {BERT_MODEL}')

Tokenizer loaded: distilbert-base-multilingual-cased


In [9]:
class BertClaimDataset(Dataset):
    def __init__(self, texts_list, labels, tokenizer, max_len=BERT_MAX_LEN):
        self.encodings = tokenizer(
            texts_list, truncation=True, padding='max_length',
            max_length=max_len, return_tensors='pt'
        )
        self.labels = torch.LongTensor(labels)
    
    def __len__(self): return len(self.labels)
    
    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item


def compute_metrics_hf(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    f1_m = f1_score(labels, preds, average='macro')
    f1_d = f1_score(labels, preds, pos_label=0)
    return {'f1_macro': f1_m, 'f1_declined': f1_d}

In [ ]:
def bert_fn(X_data, y_data, tr_idx, va_idx):
    """Train BERT and return val predictions."""
    texts_arr = np.array(texts)
    tr_texts = texts_arr[tr_idx].tolist()
    va_texts = texts_arr[va_idx].tolist()
    y_tr, y_va = y_data[tr_idx], y_data[va_idx]
    
    # Compute class weights
    n_neg = (y_tr == 0).sum()
    n_pos = (y_tr == 1).sum()
    weight_for_0 = len(y_tr) / (2 * n_neg)
    weight_for_1 = len(y_tr) / (2 * n_pos)
    class_weights = torch.FloatTensor([weight_for_0, weight_for_1]).to(device)
    
    train_ds = BertClaimDataset(tr_texts, y_tr, tokenizer)
    val_ds = BertClaimDataset(va_texts, y_va, tokenizer)
    
    model = AutoModelForSequenceClassification.from_pretrained(
        BERT_MODEL, num_labels=2
    ).to(device)
    
    # Custom trainer with class weights
    class WeightedTrainer(Trainer):
        def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
            labels = inputs.pop('labels')
            outputs = model(**inputs)
            logits = outputs.logits
            loss_fn = nn.CrossEntropyLoss(weight=class_weights)
            loss = loss_fn(logits, labels)
            return (loss, outputs) if return_outputs else loss
    
    training_args = TrainingArguments(
        output_dir='./bert_tmp',
        num_train_epochs=BERT_EPOCHS,
        per_device_train_batch_size=BERT_BATCH,
        per_device_eval_batch_size=BERT_BATCH * 2,
        learning_rate=BERT_LR,
        weight_decay=0.01,
        eval_strategy='epoch',
        save_strategy='no',
        logging_steps=50,
        disable_tqdm=True,
        report_to='none',
        seed=RANDOM_STATE,
    )
    
    trainer = WeightedTrainer(
        model=model, args=training_args,
        train_dataset=train_ds, eval_dataset=val_ds,
        compute_metrics=compute_metrics_hf,
    )
    trainer.train()
    
    # Predict
    preds = trainer.predict(val_ds)
    logits = preds.predictions
    proba = torch.softmax(torch.FloatTensor(logits), dim=-1)[:, 1].numpy()
    y_pred = np.argmax(logits, axis=-1)
    
    # Cleanup
    del model, trainer
    torch.cuda.empty_cache() if torch.cuda.is_available() else None
    
    return y_pred, proba

print('=== Approach C: Fine-tuned BERT (text only) ===')
print(f'Model: {BERT_MODEL}, Epochs: {BERT_EPOCHS}, LR: {BERT_LR}')
bert_results = cv_collect(bert_fn, None, y, f'Fine-tuned BERT ({BERT_MODEL})')

=== Approach C: Fine-tuned BERT (text only) ===
Model: distilbert-base-multilingual-cased, Epochs: 3, LR: 2e-05
This will take several minutes on CPU...



Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-multilingual-cased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{'loss': '0.6916', 'grad_norm': '1.381', 'learning_rate': '1.773e-05', 'epoch': '0.3472'}
{'loss': '0.7024', 'grad_norm': '2.496', 'learning_rate': '1.542e-05', 'epoch': '0.6944'}
{'eval_loss': '0.6763', 'eval_f1_macro': '0.5477', 'eval_f1_declined': '0.2179', 'eval_runtime': '21.68', 'eval_samples_per_second': '26.57', 'eval_steps_per_second': '0.83', 'epoch': '1'}
{'loss': '0.6851', 'grad_norm': '1.633', 'learning_rate': '1.31e-05', 'epoch': '1.042'}
{'loss': '0.6879', 'grad_norm': '2.911', 'learning_rate': '1.079e-05', 'epoch': '1.389'}
{'loss': '0.6593', 'grad_norm': '4.917', 'learning_rate': '8.472e-06', 'epoch': '1.736'}
{'eval_loss': '0.6773', 'eval_f1_macro': '0.6028', 'eval_f1_declined': '0.2946', 'eval_runtime': '22.72', 'eval_samples_per_second': '25.35', 'eval_steps_per_second': '0.792', 'epoch': '2'}
{'loss': '0.6552', 'grad_norm': '5.353', 'learning_rate': '6.157e-06', 'epoch': '2.083'}
{'loss': '0.6001', 'grad_norm': '3.215', 'learning_rate': '3.843e-06', 'epoch': '2.431

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-multilingual-cased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{'loss': '0.6976', 'grad_norm': '1.303', 'learning_rate': '1.773e-05', 'epoch': '0.3472'}
{'loss': '0.6848', 'grad_norm': '2.055', 'learning_rate': '1.542e-05', 'epoch': '0.6944'}
{'eval_loss': '0.6976', 'eval_f1_macro': '0.4634', 'eval_f1_declined': '0.02', 'eval_runtime': '26.13', 'eval_samples_per_second': '22.05', 'eval_steps_per_second': '0.689', 'epoch': '1'}
{'loss': '0.6815', 'grad_norm': '4.028', 'learning_rate': '1.31e-05', 'epoch': '1.042'}


---
## 6. Approach D: BERT Embeddings + Structural Features → LightGBM (Hybrid)

Extract CLS token embeddings from the fine-tuned BERT, combine with structural tabular features, feed into LightGBM. Best of both worlds.

In [ ]:
def bert_hybrid_fn(X_data, y_data, tr_idx, va_idx):
    """Fine-tune BERT, extract CLS embeddings, combine with structural, train LightGBM."""
    texts_arr = np.array(texts)
    tr_texts = texts_arr[tr_idx].tolist()
    va_texts = texts_arr[va_idx].tolist()
    y_tr, y_va = y_data[tr_idx], y_data[va_idx]
    
    # Class weights
    n_neg, n_pos = (y_tr == 0).sum(), (y_tr == 1).sum()
    class_weights = torch.FloatTensor(
        [len(y_tr)/(2*n_neg), len(y_tr)/(2*n_pos)]
    ).to(device)
    
    train_ds = BertClaimDataset(tr_texts, y_tr, tokenizer)
    val_ds = BertClaimDataset(va_texts, y_va, tokenizer)
    
    model = AutoModelForSequenceClassification.from_pretrained(
        BERT_MODEL, num_labels=2
    ).to(device)
    
    class WeightedTrainer(Trainer):
        def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
            labels = inputs.pop('labels')
            outputs = model(**inputs)
            loss = nn.CrossEntropyLoss(weight=class_weights)(outputs.logits, labels)
            return (loss, outputs) if return_outputs else loss
    
    training_args = TrainingArguments(
        output_dir='./bert_tmp_hybrid',
        num_train_epochs=BERT_EPOCHS,
        per_device_train_batch_size=BERT_BATCH,
        per_device_eval_batch_size=BERT_BATCH * 2,
        learning_rate=BERT_LR,
        weight_decay=0.01,
        save_strategy='no',
        logging_steps=50,
        disable_tqdm=True,
        report_to='none',
        seed=RANDOM_STATE,
    )
    
    trainer = WeightedTrainer(
        model=model, args=training_args, train_dataset=train_ds,
    )
    trainer.train()
    
    # Extract CLS embeddings from fine-tuned model
    model.eval()
    def extract_embeddings(dataset):
        loader = DataLoader(dataset, batch_size=BERT_BATCH * 2, shuffle=False)
        all_embs = []
        with torch.no_grad():
            for batch in loader:
                inputs = {k: v.to(device) for k, v in batch.items() if k != 'labels'}
                outputs = model.distilbert(**inputs)
                # CLS token embedding (first token)
                cls_emb = outputs.last_hidden_state[:, 0, :].cpu().numpy()
                all_embs.append(cls_emb)
        return np.vstack(all_embs)
    
    emb_tr = extract_embeddings(train_ds)
    emb_va = extract_embeddings(val_ds)
    
    # Combine with structural features
    X_combined_tr = np.hstack([X_struct[tr_idx], emb_tr])
    X_combined_va = np.hstack([X_struct[va_idx], emb_va])
    
    # Train LightGBM on combined features
    lgb_model = lgb.LGBMClassifier(
        n_estimators=500, max_depth=5, learning_rate=0.05,
        is_unbalance=True, random_state=RANDOM_STATE, n_jobs=-1, verbose=-1,
    )
    lgb_model.fit(X_combined_tr, y_tr)
    y_pred = lgb_model.predict(X_combined_va)
    y_proba = lgb_model.predict_proba(X_combined_va)[:, 1]
    
    del model, trainer
    torch.cuda.empty_cache() if torch.cuda.is_available() else None
    
    return y_pred, y_proba

print('=== Approach D: BERT Hybrid (BERT emb + structural → LightGBM) ===')
print('This will take several minutes on CPU...\n')
bert_hybrid_results = cv_collect(bert_hybrid_fn, None, y, 'BERT Hybrid (CLS + structural → LGB)')

---
## 7. Full Comparison — All Attempts

In [ ]:
# Load previous results
prev = pd.read_csv(Path('../../models/model_comparison_v1_v2.csv'), index_col=0)

# v3 results
v3_entries = [
    ('v3: TF-IDF + SVD + LGB', tfidf_results),
    ('v3: BiLSTM (text only)', lstm_results),
    (f'v3: BERT fine-tuned (text)', bert_results),
    ('v3: BERT hybrid + LGB', bert_hybrid_results),
]

v3_rows = []
for name, res in v3_entries:
    mdf = res['metrics_df']
    v3_rows.append({
        'Model': name,
        'F1 Macro (mean)': mdf['f1_macro'].mean(),
        'F1 Macro (std)': mdf['f1_macro'].std(),
        'F1 Declined (mean)': mdf['f1_declined'].mean(),
        'PR-AUC (mean)': mdf['pr_auc'].mean(),
        'ROC-AUC (mean)': mdf['roc_auc'].mean(),
    })
v3_df = pd.DataFrame(v3_rows).set_index('Model').round(4)

all_comparison = pd.concat([prev, v3_df])

print('=== FULL MODEL COMPARISON (v1 / v2 / v3) ===')
print(all_comparison.to_string())

In [ ]:
# Visual: v3 approaches only
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, metric in zip(axes, ['F1 Macro (mean)', 'F1 Declined (mean)', 'PR-AUC (mean)']):
    v3_df[metric].plot(kind='bar', ax=ax, color=['#4C78A8', '#E45756', '#54A24B', '#F58518'], edgecolor='black')
    ax.set_title(metric)
    ax.set_ylim(0, 1)
    ax.tick_params(axis='x', rotation=35)
    for p in ax.patches:
        ax.annotate(f'{p.get_height():.3f}', (p.get_x() + p.get_width()/2., p.get_height()),
                    ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.suptitle('Attempt #3 — NLP Model Comparison', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Best model across ALL attempts
best_overall = all_comparison['F1 Macro (mean)'].idxmax()
print(f'Best model overall: {best_overall}')
print(all_comparison.loc[best_overall])

best_declined = all_comparison['F1 Declined (mean)'].idxmax()
print(f'\nBest on Declined: {best_declined}')
print(all_comparison.loc[best_declined])

---
## 8. Threshold Optimization on Best v3 Model

In [ ]:
# Find best v3 model by F1 Macro
v3_all = {
    'TF-IDF': tfidf_results,
    'BiLSTM': lstm_results,
    'BERT text': bert_results,
    'BERT hybrid': bert_hybrid_results,
}
best_v3_key = max(v3_all, key=lambda k: v3_all[k]['metrics_df']['f1_macro'].mean())
best_v3 = v3_all[best_v3_key]
print(f'Threshold tuning on: {best_v3_key}')

y_true = best_v3['y_true']
y_proba = best_v3['y_proba']

thresholds = np.arange(0.1, 0.95, 0.01)
f1_macros = [f1_score(y_true, (y_proba >= t).astype(int), average='macro') for t in thresholds]
f1_declineds = [f1_score(y_true, (y_proba >= t).astype(int), pos_label=0) for t in thresholds]
recall_dec = [(y_true[(y_proba < t)] == 0).sum() / max((y_true == 0).sum(), 1) for t in thresholds]

best_t_macro = thresholds[np.argmax(f1_macros)]
best_t_declined = thresholds[np.argmax(f1_declineds)]

print(f'Best threshold F1 Macro:    {best_t_macro:.2f} -> F1={max(f1_macros):.4f}')
print(f'Best threshold F1 Declined: {best_t_declined:.2f} -> F1={max(f1_declineds):.4f}')

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(thresholds, f1_macros, label='F1 Macro', color='steelblue', linewidth=2)
ax.plot(thresholds, f1_declineds, label='F1 Declined', color='salmon', linewidth=2)
ax.plot(thresholds, recall_dec, label='Recall Declined', color='green', linewidth=2, linestyle='--')
ax.axvline(x=0.5, color='gray', linestyle='--', alpha=0.5, label='Default')
ax.axvline(x=best_t_macro, color='steelblue', linestyle=':', label=f'Best Macro ({best_t_macro:.2f})')
ax.set_xlabel('Threshold'); ax.set_ylabel('Score')
ax.set_title('Threshold Optimization — Best v3 Model'); ax.legend()
plt.tight_layout(); plt.show()

print(f'\n=== Report at Optimized Threshold ({best_t_macro:.2f}) ===')
y_pred_opt = (y_proba >= best_t_macro).astype(int)
print(classification_report(y_true, y_pred_opt, target_names=['Declined', 'Completed']))

---
## 9. Save Artifacts

In [ ]:
output_dir = Path('../../models')

# Save full comparison
all_comparison.to_csv(output_dir / 'model_comparison_v1_v2_v3.csv')
print(f'Saved: model_comparison_v1_v2_v3.csv')

# Save v3 config
v3_config = {
    'best_v3_model': best_v3_key,
    'optimal_threshold': float(best_t_macro),
    'bert_model': BERT_MODEL,
    'bert_epochs': BERT_EPOCHS,
    'bert_lr': BERT_LR,
    'bert_max_len': BERT_MAX_LEN,
    'tfidf_svd_components': 80,
    'lstm_max_vocab': MAX_VOCAB,
    'lstm_max_len': MAX_LEN,
    'structural_features': structural_features,
    'v3_results': {name: res['metrics'] for name, res in v3_entries},
}
with open(output_dir / 'model_config_v3.json', 'w') as f:
    json.dump(v3_config, f, indent=2)
print(f'Saved: model_config_v3.json')

# If BERT hybrid is best, retrain and save
# (For the final model, we'll pick the best approach in the summary)
print(f'\nBest v3 approach: {best_v3_key}')
print('Final model will be retrained on full data in the production pipeline.')

---
## Wrap up

Tried 4 approaches from simplest to most complex:
- TF-IDF was surprisingly competitive (0.591) - specific words do matter
- BiLSTM was the worst (0.551) - not enough data to learn from scratch, as expected
- BERT fine-tuned got best F1 declined (0.365) and 46% recall on declined claims
- BERT hybrid (CLS embeddings + structural features -> LightGBM) got best F1 macro (0.608) and was the most stable across folds

BERT helps but fine-tuning takes forever on CPU. The gains over v2 are real but diminishing. Going to try LLM-based feature extraction next since the text signal is clearly the bottleneck.